In [1]:
import sys
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm

import ollama  # pip install ollama
from sentence_transformers import SentenceTransformer

# Nazwa modelu lokalnego w Ollama – uruchom wcześniej: ollama pull gemma4:12b
OLLAMA_MODEL = "gemma4:e4b"

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# paraphrase-multilingual-mpnet-base-v2 produkuje wektory o wymiarze 768
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

print(f"Embedder gotowy. Wymiar wektora: {embedder.get_embedding_dimension()}")
print(f"Model LLM: {OLLAMA_MODEL} (obsługiwany przez Ollama)")


Embedder gotowy. Wymiar wektora: 768
Model LLM: gemma4:e4b (obsługiwany przez Ollama)


In [2]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_sentence_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal)  # 0


Number of chunks:  0


In [3]:
class Utils:
    def __init__(self, embedding_model: SentenceTransformer = None,
                 ollama_model: str = None,
                 index=None, metadata=None, chunk_size=256):
        self.embedding_model = embedding_model
        self.ollama_model = ollama_model  # nazwa modelu w Ollama (zamiast llm_model + llm_tokenizer)
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        os.makedirs(db_loc, exist_ok=True)
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)

    def answer_question(self, prompt_template="", query="", max_tokens=4096, temp=0.7, k=12):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        # Budujemy historię rozmowy – Ollama przyjmuje ten sam format messages co OpenAI
        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        # Wywołanie lokalnego modelu przez Ollama (zamiast transformers model.generate)
        # Ollama musi działać w tle: uruchom 'ollama serve' lub aplikację Ollama
        response = ollama.chat(
            model=self.ollama_model,
            messages=messages,
            options={
                "temperature": temp,       # im niższa tym bardziej deterministyczna odpowiedź
                "num_predict": max_tokens  # maksymalna liczba nowych tokenów do wygenerowania
            }
        )

        # Wyciągamy tekst odpowiedzi ze struktury zwróconej przez Ollama
        answer = response.message.content

        return answer, chunks


In [4]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedder,        # model embeddingowy do zamiany tekstu na wektory
    OLLAMA_MODEL,    # nazwa modelu Ollama (zamiast model + tokenizer)
    index,           # indeks FAISS z wektorami chunków
    metadata,        # metadane chunków (tekst, strona, plik)
    chunk_size=512   # ilość treści w każdym fragmencie - zwiększamy by dać modelowi więcej treści gdy odmówi odpowiedzi
)


In [5]:
knowledge_dir = "knowledge/"
# Przetwarzamy każdy plik PDF z katalogu – dzielimy na chunki i dodajemy do FAISS
for file in os.listdir(knowledge_dir):
    utils.process_file(knowledge_dir + file)

print('Number of chunks: ', index.ntotal)


Adding chunks to FAISS: 100%|██████████| 880/880 [00:52<00:00, 16.76it/s]


Number of chunks:  1541


In [6]:
# przykładowe pytania dotyczące powyższych dokumentów
questions = [
    # HerbAtlas (herbatlas_eng-2.pdf)
    "Jakie są właściwości lecznicze aloesu?",
    "Jak imbir jest stosowany w tradycyjnej medycynie?",
    "Jakie są wymagania uprawowe rumianku?",

    # Fizjologia roślin - Vince Ordóg (plant-physiology-vince-ordog-3.pdf)
    "Czym jest potencjał wodny roślin i jakie czynniki na niego wpływają?",
    "Jak hormony roślinne auksyny wpływają na wzrost rośliny?",
    "Co się dzieje z roślinami podczas stresu temperaturowego?",
    "Na czym polega fotosynteza u roślin C4?",

    # Wstęp do botaniki - Shipunov (introduction-to-botany-alexey-shipunov-892.pdf)
    "Jaka jest różnica między ksylemem a floemem?",
    "Czym różni się mitoza od mejozy u roślin?",
    "Jakie są główne typy tkanek roślinnych?",
]


In [7]:
from IPython.display import display, Markdown

prompt_template = """Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: 
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)


response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=random_query,
    max_tokens=4096,
    temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jaka jest różnica między ksylemem a floemem?

**Odpowiedź:**

Elementy ksylemu i floemu różnią się strukturą i składem komórkowym.

**Ksylem:**
*   Składa się z elementów takich jak naczynia (vessels) oraz tracheidy.
*   Elementy ksylemu, z wyjątkiem parenchymy, są bogate w ligninę i stanowią główne składniki drewna.
*   Tracheidy mają zamknięte końce i łączą się za pomocą otworów (pits), natomiast elementy naczyń są bardziej lub mniej otwarte i łączą się przez perforacje.

**Floem:**
*   Charakterystyczną cechą tkanki splotowej jest komórka przewodząca zwana elementem siewnym (sieve element) lub rurką siewną (sieve tube).
*   Element siewny to wydłużony szereg indywidualnych komórek, zwanych członami rurki siewnej.
*   W przeciwieństwie do elementów tracheidowych ksylemu, elementy siewne nie mają sztywnych ścian i zawierają żywe protoplasty w dojrzałym stanie funkcjonalnym.

---
**Źródła:**

**[1]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 30

> (difference of potentials) between thylakoid (vesicle or membrane pocket) and ma- trix (stroma) compartments of the chloroplast (Fig. 2.5). To make this difference, the cell needs to segregate ions:...

**[2]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 175

> These ﬂoristic differences are due to the various geological and biological histories of these places. Plant biogeography studies them, explains them and creates the ﬂoristic kingdoms classiﬁcation (F...

...

**[8]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 72

> Cell types and tissues “Parenchyma” and “sclerenchyma” terms are freShoot systemquently used in two ways: ﬁrst, to name tissues (or even classes of tissues) which occur in multiple places of the plant...

**[9]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 160

> Figure 8.12. Comparison of Archaefructus ﬂower (left) and typical ﬂower (note colors). (Mod- iﬁed from various sources.) 8.2.2 The Inﬂorescence Inﬂorescence is an isolated generative shoot (shoot bear...

**[10]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 74

> bium) consists mostly of vessels with open perforations. The common name for sec- ondary xylem is wood. It is a mistake to think that tracheids are better than vessels. In fact, the main prob- lem is...

**[11]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 69

> Figure 5.2. Phagocytella (proto-animal) with kinoblast and phagocytoblast vs. proto-plant with epidermis and ground tissue. Collenchyma (Fig.5.4) is living supportive tissue that has elongated cells a...

**[12]** `plant-physiology-vince-ordog.pdf` — strona 47

> Production of primary and secondary  metabolites      41    Created by XMLmind XSL-FO Converter.  and a transmembrane electric potential. Transmembrane pH difference of one pH unit is equivalent to...

In [8]:
random_query = "Jak bardzo radioaktywne są banany"
display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jak bardzo radioaktywne są banany

**Odpowiedź:**

Elementy ksylemu i floemu różnią się strukturą i składem komórkowym.

**Ksylem:**
*   Składa się z elementów takich jak naczynia (vessels) oraz tracheidy.
*   Elementy ksylemu, z wyjątkiem parenchymy, są bogate w ligninę i stanowią główne składniki drewna.
*   Tracheidy mają zamknięte końce i łączą się za pomocą otworów (pits), natomiast elementy naczyń są bardziej lub mniej otwarte i łączą się przez perforacje.

**Floem:**
*   Charakterystyczną cechą tkanki splotowej jest komórka przewodząca zwana elementem siewnym (sieve element) lub rurką siewną (sieve tube).
*   Element siewny to wydłużony szereg indywidualnych komórek, zwanych członami rurki siewnej.
*   W przeciwieństwie do elementów tracheidowych ksylemu, elementy siewne nie mają sztywnych ścian i zawierają żywe protoplasty w dojrzałym stanie funkcjonalnym.

---
**Źródła:**

**[1]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 30

> (difference of potentials) between thylakoid (vesicle or membrane pocket) and ma- trix (stroma) compartments of the chloroplast (Fig. 2.5). To make this difference, the cell needs to segregate ions:...

**[2]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 175

> These ﬂoristic differences are due to the various geological and biological histories of these places. Plant biogeography studies them, explains them and creates the ﬂoristic kingdoms classiﬁcation (F...

...

**[8]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 72

> Cell types and tissues “Parenchyma” and “sclerenchyma” terms are freShoot systemquently used in two ways: ﬁrst, to name tissues (or even classes of tissues) which occur in multiple places of the plant...

**[9]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 160

> Figure 8.12. Comparison of Archaefructus ﬂower (left) and typical ﬂower (note colors). (Mod- iﬁed from various sources.) 8.2.2 The Inﬂorescence Inﬂorescence is an isolated generative shoot (shoot bear...

**[10]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 74

> bium) consists mostly of vessels with open perforations. The common name for sec- ondary xylem is wood. It is a mistake to think that tracheids are better than vessels. In fact, the main prob- lem is...

**[11]** `introduction-to-botany-alexey-shipunov-892.pdf` — strona 69

> Figure 5.2. Phagocytella (proto-animal) with kinoblast and phagocytoblast vs. proto-plant with epidermis and ground tissue. Collenchyma (Fig.5.4) is living supportive tissue that has elongated cells a...

**[12]** `plant-physiology-vince-ordog.pdf` — strona 47

> Production of primary and secondary  metabolites      41    Created by XMLmind XSL-FO Converter.  and a transmembrane electric potential. Transmembrane pH difference of one pH unit is equivalent to...